# X6 Internalization — Google Colab Operator Gate

## Purpose

This notebook validates a future, source-bound X6 operator bundle on Google Colab. It is independent of Kaggle, installs nothing, loads no checkpoint, runs no model inference, and performs no training. Missing or invalid scientific artifacts are reported as blockers rather than uncaught Python exceptions.

## Optional future input directory

Attach an unzipped directory to `/content/x6_operator_inputs`, or set `X6_OPERATOR_INPUT_ROOT` before running. Required files are:

- `x1_receipt.json`
- `parent.json`
- `split_bundle.json`
- `arm_manifest.json`
- `dataset_manifests.json`
- `continuation_manifests.json`
- `evaluation_commitment.json`
- `source_release.json`
- `run_manifest.json`

The embedded protocol is still `PROPOSED_PENDING_REVIEW`, so the current expected verdict is `PREREQUISITE_BLOCKED`. A future authorized operator must use an owner-approved versioned protocol and a matching source release.

In [ ]:
from __future__ import annotations

import base64
import gzip
import hashlib
import importlib.util
import io
import json
import os
from pathlib import Path, PurePosixPath
import platform
import secrets
import stat
import subprocess
import sys
import tempfile
from datetime import datetime, timezone
import zipfile

if sys.version_info < (3, 10):
    raise RuntimeError("This notebook requires Python 3.10 or newer: " + platform.python_version())

content_override = os.environ.get("X6_COLAB_CONTENT_ROOT", "").strip()
CONTENT_ROOT = Path(content_override).expanduser().resolve() if content_override else (Path("/content") if Path("/content").is_dir() else Path(tempfile.gettempdir()) / "x6_x7_colab")
CONTENT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + secrets.token_hex(4)
RUN_ROOT = CONTENT_ROOT / "x6_x7_colab_runs" / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
SOURCE_ROOT = RUN_ROOT / "source"

EMBEDDED_BUNDLE_B64 = (
    "..."
    "QEUEAHvzSRV/CAEA"
)


def sha256_file(path: str | Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def safe_extract(archive: zipfile.ZipFile, destination: Path) -> list[str]:
    destination = destination.resolve()
    extracted: list[str] = []
    for info in archive.infolist():
        relative = PurePosixPath(info.filename)
        mode = info.external_attr >> 16
        if relative.is_absolute() or ".." in relative.parts or stat.S_ISLNK(mode):
            raise RuntimeError("Unsafe embedded archive member: " + info.filename)
        target = (destination / Path(*relative.parts)).resolve()
        try:
            target.relative_to(destination)
        except ValueError as exc:
            raise RuntimeError("Embedded archive member escapes destination") from exc
        if info.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(info, "r") as source, target.open("xb") as output:
            while True:
                block = source.read(1 << 20)
                if not block:
                    break
                output.write(block)
        extracted.append(relative.as_posix())
    return sorted(extracted)


with zipfile.ZipFile(io.BytesIO(gzip.decompress(base64.b64decode(EMBEDDED_BUNDLE_B64)))) as archive:
    extracted_files = safe_extract(archive, SOURCE_ROOT)
if len(extracted_files) != 15:
    raise RuntimeError("Embedded source bundle is incomplete")

spec = importlib.util.spec_from_file_location("anra_x6_x7_colab", SOURCE_ROOT / "x_factor" / "x6_x7.py")
if spec is None or spec.loader is None:
    raise RuntimeError("Unable to load the embedded X6/X7 contract module")
x6_x7 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = x6_x7
spec.loader.exec_module(x6_x7)
if "torch" in sys.modules or "anra_core" in sys.modules:
    raise RuntimeError("Model runtime was imported during the model-free bootstrap")

PROTOCOL_PATH = SOURCE_ROOT / "x_factor" / "protocols" / "x6_x7_v1.json"
EMBEDDED_AUDIT_PATH = SOURCE_ROOT / "x_factor" / "receipts" / "x1_to_x6_entry_audit_v1.json"
LEGACY_X1_PATH = SOURCE_ROOT / "output" / "x1_real_receipt.json"
LEGACY_IBQ_PATH = SOURCE_ROOT / "output" / "ibq_legacy_basis_verdict.json"
PROTOCOL = x6_x7.load_json(PROTOCOL_PATH)
PROTOCOL_VERDICT = x6_x7.validate_protocol(PROTOCOL, SOURCE_ROOT, check_source_closure=True)
if not PROTOCOL_VERDICT["valid"]:
    raise RuntimeError("Embedded protocol validation failed: " + repr(PROTOCOL_VERDICT["errors"]))
FRESH_X1_AUDIT = x6_x7.audit_x1_entry(
    x6_x7.load_json(LEGACY_X1_PATH),
    x6_x7.sha_file(LEGACY_X1_PATH),
)
EMBEDDED_X1_AUDIT = x6_x7.load_json(EMBEDDED_AUDIT_PATH)
if FRESH_X1_AUDIT != EMBEDDED_X1_AUDIT:
    raise RuntimeError("Embedded and recomputed X1 gate audits differ")
if FRESH_X1_AUDIT.get("status") != "BLOCKED" or FRESH_X1_AUDIT.get("verdict") != "X6_ENTRY_BLOCKED":
    raise RuntimeError("The embedded historical X1 evidence unexpectedly authorizes X6")

software_tests = {"status": "SKIPPED", "reason": "pytest is not already installed; no package installation was requested"}
if importlib.util.find_spec("pytest") is not None:
    completed = subprocess.run(
        [sys.executable, "-m", "pytest", "x_factor/tests/test_x6_x7.py", "-q"],
        cwd=SOURCE_ROOT,
        capture_output=True,
        text=True,
        timeout=120,
        check=False,
    )
    software_tests = {
        "status": "PASS" if completed.returncode == 0 else "FAIL",
        "returncode": completed.returncode,
        "stdout": completed.stdout,
        "stderr": completed.stderr,
    }
    if completed.returncode != 0:
        raise RuntimeError("Embedded focused software tests failed")

RUNTIME = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "run_id": RUN_ID,
    "run_root": str(RUN_ROOT),
    "source_root": str(SOURCE_ROOT),
    "embedded_file_count": len(extracted_files),
    "pytest": software_tests,
    "model_imported": False,
    "model_execution": False,
    "training": False,
}
print(json.dumps({"runtime": RUNTIME, "x1_gate": FRESH_X1_AUDIT}, indent=2, sort_keys=True))

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

input_override = os.environ.get("X6_OPERATOR_INPUT_ROOT", "").strip()
INPUT_ROOT = Path(input_override).expanduser().resolve() if input_override else CONTENT_ROOT / "x6_operator_inputs"
REQUIRED_FILES = (
    "x1_receipt.json",
    "parent.json",
    "split_bundle.json",
    "arm_manifest.json",
    "dataset_manifests.json",
    "continuation_manifests.json",
    "evaluation_commitment.json",
    "source_release.json",
    "run_manifest.json",
)
missing = [name for name in REQUIRED_FILES if not (INPUT_ROOT / name).is_file()]
blockers: list[str] = []
readiness = None
entry_audit = None

if missing:
    blockers.append("MISSING_OPERATOR_ARTIFACTS: " + ", ".join(missing))
else:
    try:
        x1_path = INPUT_ROOT / "x1_receipt.json"
        entry_audit = x6_x7.audit_x1_entry(
            x6_x7.load_json(x1_path),
            x6_x7.sha_file(x1_path),
        )
        parent = x6_x7.load_json(INPUT_ROOT / "parent.json")
        split_bundle = x6_x7.load_json(INPUT_ROOT / "split_bundle.json")
        arm_manifest = x6_x7.load_json(INPUT_ROOT / "arm_manifest.json")
        dataset_manifests = x6_x7.load_json(INPUT_ROOT / "dataset_manifests.json")
        continuation_manifests = x6_x7.load_json(INPUT_ROOT / "continuation_manifests.json")
        evaluation_commitment = x6_x7.load_json(INPUT_ROOT / "evaluation_commitment.json")
        source_release = x6_x7.load_json(INPUT_ROOT / "source_release.json")
        run_manifest = x6_x7.load_json(INPUT_ROOT / "run_manifest.json")
        readiness = x6_x7.assess_x6_readiness(
            PROTOCOL,
            entry_audit,
            parent,
            split_bundle,
            arm_manifest,
            continuation_manifests,
            evaluation_commitment,
            source_release,
            dataset_manifests,
            run_manifest,
        )
        blockers.extend(readiness.get("blockers", []))
    except Exception as exc:
        blockers.append(f"ARTIFACT_VALIDATION_BLOCKED: {type(exc).__name__}: {exc}")

blockers = list(dict.fromkeys(blockers))
receipt = {
    "schema": "anra-x6-colab-operator-readiness/v1",
    "status": readiness["status"] if readiness is not None else "PREREQUISITE_BLOCKED",
    "mode": "MODEL_FREE_COLAB_OPERATOR_GATE",
    "protocol_sha256": PROTOCOL["identity"]["protocol_sha256"],
    "input_root": str(INPUT_ROOT),
    "required_files": list(REQUIRED_FILES),
    "missing_files": missing,
    "entry_audit_sha256": entry_audit.get("entry_audit_sha256") if entry_audit else None,
    "readiness_sha256": readiness.get("readiness_sha256") if readiness else None,
    "blockers": blockers,
    "execution_authorized": False,
    "model_execution": False,
    "training": False,
    "scientific_result": False,
    "next_action": "Supply a complete source-bound X1/X6 artifact set after the owner freezes the amendment. This notebook will not train even if the gate passes; a separately authorized execution notebook is required.",
}
receipt_path = RUN_ROOT / "x6_colab_operator_readiness.json"
receipt_path.write_text(json.dumps(receipt, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({"status": receipt["status"], "blockers": blockers, "receipt": str(receipt_path)}, indent=2, sort_keys=True))

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None
if colab_files is not None:
    colab_files.download(str(receipt_path))